In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
from pathlib import Path
import numpy as np
import pandas as pd

DATASET_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset"
)

CLIP_MANIFEST = (
    DATASET_ROOT
    / "metadata"
    / "03_clip_info"
    / "clip_manifest.csv"
)

AI05_META = (
    DATASET_ROOT
    / "metadata"
    / "05_split_info"
)

AI05_META.mkdir(
    parents=True,
    exist_ok=True,
)

SPLIT_MANIFEST = (
    AI05_META
    / "split_manifest.csv"
)

SPLIT_RATIOS = {
    "TRAIN": 0.70,
    "VALIDATION": 0.15,
    "TEST": 0.15,
}

SPLIT_NAMES = [
    "TRAIN",
    "VALIDATION",
    "TEST",
]

RANDOM_SEED = 2026

In [32]:
manifest = pd.read_csv(
    CLIP_MANIFEST
)

print("Clips:", len(manifest))
print("Groups:", manifest["provenance_group_id"].nunique())

print()
print("Classes:")
print(manifest["class_name"].value_counts())

print()
print("Datasets:")
print(manifest["source_dataset"].value_counts())

Clips: 7749
Groups: 4241

Classes:
class_name
COUGH    5122
OTHER    1520
SNORE    1107
Name: count, dtype: int64

Datasets:
source_dataset
Coswara            5082
ESC-50             1600
AudioSet Strong    1067
Name: count, dtype: int64


In [33]:
# class*datasetのstratumを作る

manifest["stratum"] = (
    manifest["source_dataset"].astype(str)
    + "__"
    + manifest["class_name"].astype(str)
)

print(
    manifest["stratum"]
    .value_counts()
)

stratum
Coswara__COUGH            5082
ESC-50__OTHER             1520
AudioSet Strong__SNORE    1067
ESC-50__SNORE               40
ESC-50__COUGH               40
Name: count, dtype: int64


In [34]:
# stratumごとにgroup一覧を表示

group_table = (
    manifest
    .groupby(
        [
            "provenance_group_id",
            "stratum",
        ]
    )
    .size()
    .reset_index(name="num_clips")
)

print(
    group_table["num_clips"]
    .describe()
)

print()
print(
    group_table["stratum"]
    .value_counts()
)

count    4241.000000
mean        1.827163
std         0.774948
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max        10.000000
Name: num_clips, dtype: float64

stratum
Coswara__COUGH            2601
ESC-50__OTHER             1185
AudioSet Strong__SNORE     383
ESC-50__COUGH               39
ESC-50__SNORE               33
Name: count, dtype: int64


In [35]:
# stratum単位のgreedy関数

def assign_stratum_groups(
    stratum_groups,
    random_seed,
):
    """
    provenance groupを壊さず、
    1 stratum内を70/15/15へgreedy分割する。
    """

    total_clips = int(
        stratum_groups["num_clips"].sum()
    )

    targets = {
        split:
            total_clips * ratio
        for split, ratio
        in SPLIT_RATIOS.items()
    }

    current = {
        split: 0
        for split in SPLIT_NAMES
    }

    # 同じ大きさのgroupについてのみ
    # deterministic random tie breaking
    rng = np.random.default_rng(
        random_seed
    )

    work = stratum_groups.copy()

    work["random_order"] = rng.random(
        len(work)
    )

    # 大きいgroupから処理
    work = work.sort_values(
        by=[
            "num_clips",
            "random_order",
        ],
        ascending=[
            False,
            True,
        ],
    )

    assignments = {}

    for _, row in work.iterrows():

        group_id = row[
            "provenance_group_id"
        ]

        group_size = int(
            row["num_clips"]
        )

        scores = {}

        for candidate_split in SPLIT_NAMES:

            hypothetical = (
                current.copy()
            )

            hypothetical[
                candidate_split
            ] += group_size

            # このstratumの3 split全体を評価する
            error = 0.0

            for split in SPLIT_NAMES:

                diff = (
                    hypothetical[split]
                    - targets[split]
                )

                error += diff * diff

            scores[
                candidate_split
            ] = error

        best_score = min(
            scores.values()
        )

        best_splits = [
            split
            for split, score
            in scores.items()
            if np.isclose(
                score,
                best_score,
            )
        ]

        # 完全tieの場合は
        # targetに対する不足数が最大のsplitへ
        if len(best_splits) > 1:

            best_split = max(
                best_splits,
                key=lambda split:
                    (
                        targets[split]
                        - current[split]
                    ),
            )

        else:
            best_split = (
                best_splits[0]
            )

        assignments[
            group_id
        ] = best_split

        current[
            best_split
        ] += group_size

    return (
        assignments,
        current,
        targets,
    )

In [36]:
# 全てのstratumを処理

all_assignments = {}

split_summary = []

for i, stratum in enumerate(
    sorted(
        group_table["stratum"].unique()
    )
):

    subset = group_table[
        group_table["stratum"]
        == stratum
    ].copy()

    assignments, counts, targets = (
        assign_stratum_groups(
            subset,
            RANDOM_SEED + i,
        )
    )

    all_assignments.update(
        assignments
    )

    split_summary.append({
        "stratum":
            stratum,

        "total":
            int(
                subset[
                    "num_clips"
                ].sum()
            ),

        "train":
            counts["TRAIN"],

        "validation":
            counts["VALIDATION"],

        "test":
            counts["TEST"],

        "target_train":
            targets["TRAIN"],

        "target_validation":
            targets["VALIDATION"],

        "target_test":
            targets["TEST"],
    })


split_summary = pd.DataFrame(
    split_summary
)

print(
    split_summary.to_string(
        index=False
    )
)

               stratum  total  train  validation  test  target_train  target_validation  target_test
AudioSet Strong__SNORE   1067    747         160   160         746.9             160.05       160.05
        Coswara__COUGH   5082   3558         762   762        3557.4             762.30       762.30
         ESC-50__COUGH     40     28           6     6          28.0               6.00         6.00
         ESC-50__OTHER   1520   1064         228   228        1064.0             228.00       228.00
         ESC-50__SNORE     40     28           6     6          28.0               6.00         6.00


In [37]:
# manifestへ反映

split_manifest = (
    manifest.copy()
)

split_manifest["split"] = (
    split_manifest[
        "provenance_group_id"
    ].map(
        all_assignments
    )
)

assert (
    split_manifest["split"]
    .notna()
    .all()
)

print(
    split_manifest["split"]
    .value_counts()
)

split
TRAIN         5425
VALIDATION    1162
TEST          1162
Name: count, dtype: int64


#### 検証

In [38]:
group_split_counts = (
    split_manifest
    .groupby(
        "provenance_group_id"
    )["split"]
    .nunique()
)

leaked_groups = (
    group_split_counts[
        group_split_counts > 1
    ]
)

print(
    "Leaked provenance groups:",
    len(leaked_groups)
)

assert len(leaked_groups) == 0

print(
    "Provenance leakage check: OK"
)

Leaked provenance groups: 0
Provenance leakage check: OK


In [39]:
print("=== Split counts ===")

split_counts = (
    split_manifest["split"]
    .value_counts()
    .reindex(SPLIT_NAMES)
)

print(split_counts)

print()
print("=== Split ratio ===")

print(
    split_counts
    / len(split_manifest)
)

=== Split counts ===
split
TRAIN         5425
VALIDATION    1162
TEST          1162
Name: count, dtype: int64

=== Split ratio ===
split
TRAIN         0.700090
VALIDATION    0.149955
TEST          0.149955
Name: count, dtype: float64


In [40]:
class_table = pd.crosstab(
    split_manifest["class_name"],
    split_manifest["split"],
)

print(
    "=== Class counts ==="
)

print(class_table)

print()
print(
    "=== Split ratio within each class ==="
)

class_ratio = (
    class_table
    .div(
        class_table.sum(axis=1),
        axis=0,
    )
)

print(class_ratio)

=== Class counts ===
split       TEST  TRAIN  VALIDATION
class_name                         
COUGH        768   3586         768
OTHER        228   1064         228
SNORE        166    775         166

=== Split ratio within each class ===
split           TEST     TRAIN  VALIDATION
class_name                                
COUGH       0.149941  0.700117    0.149941
OTHER       0.150000  0.700000    0.150000
SNORE       0.149955  0.700090    0.149955


In [41]:
dataset_table = pd.crosstab(
    split_manifest[
        "source_dataset"
    ],
    split_manifest["split"],
)

print(
    "=== Dataset counts ==="
)

print(dataset_table)

print()
print(
    "=== Split ratio within each dataset ==="
)

dataset_ratio = (
    dataset_table
    .div(
        dataset_table.sum(axis=1),
        axis=0,
    )
)

print(dataset_ratio)

=== Dataset counts ===
split            TEST  TRAIN  VALIDATION
source_dataset                          
AudioSet Strong   160    747         160
Coswara           762   3558         762
ESC-50            240   1120         240

=== Split ratio within each dataset ===
split                TEST     TRAIN  VALIDATION
source_dataset                                 
AudioSet Strong  0.149953  0.700094    0.149953
Coswara          0.149941  0.700118    0.149941
ESC-50           0.150000  0.700000    0.150000


In [42]:
stratum_table = pd.crosstab(
    split_manifest["stratum"],
    split_manifest["split"],
)

print(
    "=== Class x Dataset counts ==="
)

print(stratum_table)

print()
print(
    "=== Split ratio within each stratum ==="
)

stratum_ratio = (
    stratum_table
    .div(
        stratum_table.sum(axis=1),
        axis=0,
    )
)

print(stratum_ratio)

=== Class x Dataset counts ===
split                   TEST  TRAIN  VALIDATION
stratum                                        
AudioSet Strong__SNORE   160    747         160
Coswara__COUGH           762   3558         762
ESC-50__COUGH              6     28           6
ESC-50__OTHER            228   1064         228
ESC-50__SNORE              6     28           6

=== Split ratio within each stratum ===
split                       TEST     TRAIN  VALIDATION
stratum                                               
AudioSet Strong__SNORE  0.149953  0.700094    0.149953
Coswara__COUGH          0.149941  0.700118    0.149941
ESC-50__COUGH           0.150000  0.700000    0.150000
ESC-50__OTHER           0.150000  0.700000    0.150000
ESC-50__SNORE           0.150000  0.700000    0.150000


In [43]:
target_ratio = pd.Series({
    "TRAIN": 0.70,
    "VALIDATION": 0.15,
    "TEST": 0.15,
})

max_deviation = (
    stratum_ratio[
        target_ratio.index
    ]
    .sub(
        target_ratio,
        axis=1,
    )
    .abs()
    .max()
    .max()
)

print(
    "Maximum stratum ratio deviation:",
    max_deviation
)

Maximum stratum ratio deviation: 0.00011806375442746653


#### debug

In [20]:
group_stratum_counts = (
    manifest
    .groupby("provenance_group_id")["stratum"]
    .nunique()
)

print(
    group_stratum_counts.value_counts()
)

multi_stratum_groups = group_stratum_counts[
    group_stratum_counts > 1
]

print(
    "Groups spanning multiple strata:",
    len(multi_stratum_groups)
)

assert len(multi_stratum_groups) == 0

print("Single-stratum provenance check: OK")

stratum
1    4241
Name: count, dtype: int64
Groups spanning multiple strata: 0
Single-stratum provenance check: OK


In [44]:
# 保存

split_manifest.to_csv(
    SPLIT_MANIFEST,
    index=False,
)

print("Saved:")
print(SPLIT_MANIFEST)

Saved:
/content/drive/MyDrive/TRON/AI/dataset/metadata/05_split_info/split_manifest.csv


In [45]:
saved_manifest = pd.read_csv(
    SPLIT_MANIFEST
)

assert len(saved_manifest) == 7749

assert (
    saved_manifest["split"]
    .value_counts()
    .to_dict()
    == {
        "TRAIN": 5425,
        "VALIDATION": 1162,
        "TEST": 1162,
    }
)

assert (
    saved_manifest
    .groupby("provenance_group_id")["split"]
    .nunique()
    .max()
    == 1
)

print("Saved split manifest check: OK")

Saved split manifest check: OK


#### mean/stdの作成

In [48]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from scipy.io import wavfile


# ============================================================
# Repository path
# ============================================================

AI_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI"
)

SRC_ROOT = AI_ROOT / "src"

if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))


# ============================================================
# Import AI preprocessing module
# ============================================================

from src.preprocess import (
    preprocess_pcm16,
    normalize_log_mel,
    NUM_MEL_BINS,
)


# ============================================================
# Dataset paths
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset"
)

OUTPUT_ROOT = (
    DATASET_ROOT
    / "processed"
    / "clips_16k_1s"
)

AI05_META = (
    DATASET_ROOT
    / "metadata"
    / "05_split_info"
)

SPLIT_MANIFEST = (
    AI05_META
    / "split_manifest.csv"
)

NORMALIZATION_CSV = (
    AI05_META
    / "normalization_stats.csv"
)

NORMALIZATION_NPZ = (
    AI05_META
    / "normalization_stats.npz"
)


# ============================================================
# Path check
# ============================================================

print("AI_ROOT       :", AI_ROOT)
print("preprocess.py :", SRC_ROOT / "preprocess.py")

assert AI_ROOT.exists()
assert SRC_ROOT.exists()
assert (SRC_ROOT / "preprocess.py").exists()

print("AI source path check: OK")

AI_ROOT       : /content/drive/MyDrive/TRON/AI
preprocess.py : /content/drive/MyDrive/TRON/AI/src/preprocess.py
AI source path check: OK


In [49]:
# trainだけ読み出す

split_manifest = pd.read_csv(
    SPLIT_MANIFEST
)

train_manifest = (
    split_manifest[
        split_manifest["split"] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Train clips:", len(train_manifest))

assert len(train_manifest) == 5425

Train clips: 5425


In [50]:
# Welford統計クラス

class RunningStats:
    """
    Welford / parallel variance algorithm.

    40 Mel binsについて、
    全TRAIN frameのmean/stdを逐次算出する。

    stdはpopulation std (ddof=0)。
    """

    def __init__(self, num_features):
        self.count = 0

        self.mean = np.zeros(
            num_features,
            dtype=np.float64,
        )

        # sum of squared deviations
        self.M2 = np.zeros(
            num_features,
            dtype=np.float64,
        )

    def update(self, x):
        """
        x:
            shape = (N, 40)

        1 clipなら通常 (98, 40)。
        """

        x = np.asarray(
            x,
            dtype=np.float64,
        )

        if x.ndim != 2:
            raise ValueError(
                f"Expected 2-D array, got {x.shape}"
            )

        if x.shape[1] != len(self.mean):
            raise ValueError(
                f"Expected {len(self.mean)} features, "
                f"got {x.shape[1]}"
            )

        batch_count = x.shape[0]

        if batch_count == 0:
            return

        batch_mean = x.mean(
            axis=0
        )

        batch_M2 = (
            (x - batch_mean) ** 2
        ).sum(
            axis=0
        )

        # 最初のbatch
        if self.count == 0:

            self.count = batch_count
            self.mean = batch_mean
            self.M2 = batch_M2

            return

        # Parallel Welford merge
        delta = (
            batch_mean
            - self.mean
        )

        new_count = (
            self.count
            + batch_count
        )

        self.mean = (
            self.mean
            + delta
            * batch_count
            / new_count
        )

        self.M2 = (
            self.M2
            + batch_M2
            + delta ** 2
            * self.count
            * batch_count
            / new_count
        )

        self.count = new_count

    def variance(self):
        """
        Population variance: ddof=0
        """

        if self.count == 0:
            raise ValueError(
                "No samples accumulated"
            )

        return (
            self.M2
            / self.count
        )

    def std(self):
        return np.sqrt(
            self.variance()
        )

In [51]:
# 逐次処理

stats = RunningStats(
    NUM_MEL_BINS
)

processed_clips = 0
processed_frames = 0

for i, row in train_manifest.iterrows():

    path = (
        OUTPUT_ROOT
        / row["relative_path"]
    )

    sr, pcm = wavfile.read(
        path
    )

    # AI-03仕様の再確認
    assert sr == 16000
    assert pcm.dtype == np.int16
    assert pcm.shape == (16000,)

    # mean/stdなし
    # → raw Log-Mel [98, 40]
    feature = preprocess_pcm16(
        pcm
    )

    assert feature.shape == (
        98,
        40,
    )

    assert np.isfinite(
        feature
    ).all()

    stats.update(
        feature
    )

    processed_clips += 1
    processed_frames += feature.shape[0]

    if processed_clips % 500 == 0:
        print(
            f"Processed: "
            f"{processed_clips} / "
            f"{len(train_manifest)} clips"
        )

print()
print(
    "Processed clips :",
    processed_clips
)

print(
    "Processed frames:",
    processed_frames
)

Processed: 500 / 5425 clips
Processed: 1000 / 5425 clips
Processed: 1500 / 5425 clips
Processed: 2000 / 5425 clips
Processed: 2500 / 5425 clips
Processed: 3000 / 5425 clips
Processed: 3500 / 5425 clips
Processed: 4000 / 5425 clips
Processed: 4500 / 5425 clips
Processed: 5000 / 5425 clips

Processed clips : 5425
Processed frames: 531650


In [52]:
# mean/stdの取得

mean = stats.mean
std = stats.std()

print("count:", stats.count)

print()
print("mean shape:", mean.shape)
print("std shape :", std.shape)

print()
print("mean:")
print(mean)

print()
print("std:")
print(std)

count: 531650

mean shape: (40,)
std shape : (40,)

mean:
[-21.98743049 -17.64521091 -16.39564074 -15.84532544 -16.13302805
 -16.43956015 -16.32274813 -15.95301555 -16.23349903 -17.40386138
 -18.33236598 -18.99581647 -19.42109553 -19.70913224 -19.68337686
 -19.70301001 -19.48438379 -19.60913755 -19.95402856 -20.66860776
 -21.47918987 -22.24971487 -22.9222613  -23.46034102 -23.64381547
 -23.70277648 -23.8402997  -23.86986337 -23.90894967 -24.50150704
 -25.64542535 -26.44344012 -26.92688431 -27.50671219 -28.30022479
 -28.98951084 -29.5626965  -30.64121222 -32.78390018 -37.08832696]

std:
[29.26804474 29.80691924 30.10273993 30.17107901 30.06205511 29.99532418
 30.14527926 30.36589225 30.34775724 30.03332352 29.74256825 29.52936787
 29.461761   29.48772095 29.55737075 29.55809916 29.56000811 29.52988297
 29.55890641 29.40112586 29.19977659 29.012857   28.81687598 28.67785222
 28.64326356 28.65473488 28.64557979 28.67714195 28.702232   28.56771921
 28.39580992 28.3728297  28.61391837 28.80

#### 検証

In [53]:
assert stats.count == (
    5425 * 98
)

assert mean.shape == (40,)
assert std.shape == (40,)

assert np.isfinite(
    mean
).all()

assert np.isfinite(
    std
).all()

assert (
    std > 0
).all()

print()
print(
    "Normalization statistics check: OK"
)


Normalization statistics check: OK


In [54]:
mean_f32 = mean.astype(
    np.float32
)

std_f32 = std.astype(
    np.float32
)

print(mean_f32.dtype)
print(std_f32.dtype)

float32
float32


In [55]:
# CSV形式で保存

normalization_df = pd.DataFrame({
    "mel_bin":
        np.arange(
            NUM_MEL_BINS
        ),

    "mean":
        mean_f32,

    "std":
        std_f32,
})

normalization_df.to_csv(
    NORMALIZATION_CSV,
    index=False,
)

print(
    normalization_df
)

print()
print(
    "Saved:",
    NORMALIZATION_CSV
)

    mel_bin       mean        std
0         0 -21.987431  29.268045
1         1 -17.645210  29.806919
2         2 -16.395641  30.102739
3         3 -15.845325  30.171080
4         4 -16.133028  30.062056
5         5 -16.439560  29.995325
6         6 -16.322748  30.145279
7         7 -15.953015  30.365892
8         8 -16.233500  30.347757
9         9 -17.403862  30.033323
10       10 -18.332365  29.742569
11       11 -18.995817  29.529367
12       12 -19.421095  29.461761
13       13 -19.709133  29.487720
14       14 -19.683376  29.557371
15       15 -19.703011  29.558100
16       16 -19.484385  29.560009
17       17 -19.609138  29.529882
18       18 -19.954029  29.558907
19       19 -20.668608  29.401125
20       20 -21.479189  29.199776
21       21 -22.249716  29.012857
22       22 -22.922262  28.816875
23       23 -23.460340  28.677853
24       24 -23.643816  28.643263
25       25 -23.702776  28.654736
26       26 -23.840300  28.645580
27       27 -23.869864  28.677141
28       28 -2

In [56]:
# npz形式で保存

np.savez(
    NORMALIZATION_NPZ,

    mean=mean_f32,
    std=std_f32,

    sample_count=np.int64(
        stats.count
    ),

    train_clip_count=np.int64(
        len(train_manifest)
    ),
)

print(
    "Saved:",
    NORMALIZATION_NPZ
)

Saved: /content/drive/MyDrive/TRON/AI/dataset/metadata/05_split_info/normalization_stats.npz


In [57]:
# 再読み込みして確認

loaded = np.load(
    NORMALIZATION_NPZ
)

loaded_mean = loaded["mean"]
loaded_std = loaded["std"]

assert np.array_equal(
    loaded_mean,
    mean_f32,
)

assert np.array_equal(
    loaded_std,
    std_f32,
)

print(
    "NPZ reload check: OK"
)

NPZ reload check: OK


#### 再検証

In [58]:
normalized_stats = RunningStats(
    NUM_MEL_BINS
)

processed = 0

for _, row in train_manifest.iterrows():

    path = (
        OUTPUT_ROOT
        / row["relative_path"]
    )

    sr, pcm = wavfile.read(
        path
    )

    feature = preprocess_pcm16(
        pcm
    )

    normalized = normalize_log_mel(
        feature,
        mean_f32,
        std_f32,
    )

    assert np.isfinite(
        normalized
    ).all()

    normalized_stats.update(
        normalized
    )

    processed += 1

    if processed % 500 == 0:
        print(
            f"Validation: "
            f"{processed} / "
            f"{len(train_manifest)}"
        )

Validation: 500 / 5425
Validation: 1000 / 5425
Validation: 1500 / 5425
Validation: 2000 / 5425
Validation: 2500 / 5425
Validation: 3000 / 5425
Validation: 3500 / 5425
Validation: 4000 / 5425
Validation: 4500 / 5425
Validation: 5000 / 5425


In [59]:
normalized_mean = (
    normalized_stats.mean
)

normalized_std = (
    normalized_stats.std()
)

print()
print(
    "Normalized mean:"
)

print(
    normalized_mean
)

print()
print(
    "Normalized std:"
)

print(
    normalized_std
)

print()
print(
    "Max |mean|:",
    np.max(
        np.abs(
            normalized_mean
        )
    )
)

print(
    "Max |std - 1|:",
    np.max(
        np.abs(
            normalized_std
            - 1.0
        )
    )
)


Normalized mean:
[ 1.15053733e-09 -2.34518011e-08  2.39306396e-08 -8.63157249e-09
 -1.12127300e-08 -6.69964311e-09  8.36404238e-10 -7.43705440e-09
  1.01087520e-08  2.37570644e-08 -2.99091752e-08  2.96599064e-08
 -1.82264640e-08  4.12375289e-08 -1.77825834e-08  1.84468619e-08
  2.42226909e-08  3.09532686e-08  1.08034377e-08  4.06187345e-09
 -3.63224366e-08  3.93045077e-08  4.03951327e-08 -6.85703395e-09
  7.69334704e-09 -2.70302095e-08  6.09404211e-09  6.48238566e-09
 -1.99312757e-08 -2.88613568e-09  1.05072875e-08 -3.16553474e-08
 -3.11035197e-08  4.06519971e-09 -1.19054039e-08  1.00765050e-08
  4.39864838e-09  6.17991362e-09  4.84092349e-08 -5.09073117e-08]

Normalized std:
[0.99999998 1.00000001 1.00000001 1.00000001 1.00000001 0.99999997
 1.00000001 0.99999998 1.00000001 1.         0.99999997 1.
 0.99999998 0.99999999 0.99999998 0.99999998 0.99999997 1.00000002
 1.00000001 1.00000002 1.00000004 0.99999997 1.         0.99999996
 1.00000005 1.         0.99999996 1.00000003 0.9999999

In [60]:
assert np.allclose(
    normalized_mean,
    0.0,
    atol=1e-5,
)

assert np.allclose(
    normalized_std,
    1.0,
    atol=1e-5,
)

print(
    "Normalized TRAIN statistics check: OK"
)

Normalized TRAIN statistics check: OK
